# IEEE PHM 2012 (PRONOSTIA) Bearing RUL — End-to-End Notebook

This notebook provides a **clean, reproducible** pipeline for the IEEE PHM 2012 bearing dataset:

1. Parse raw `acc_*.csv` and `temp_*.csv`
2. Extract robust per-record features (time + frequency)
3. Add degradation/trend features and handle missing temperature safely
4. (Optional but recommended) construct a **Health Index (HI)** and use it as a compact degradation representation
5. Build windowed time-series samples `(batch, seq_len, features)`
6. Train a strong **TCN–Transformer** model (local + global temporal modelling)
7. Evaluate with **RMSE / MAE / IEEE PHM scoring** + per-bearing plots

In [7]:
# ====== 0) Setup ======
import os, re, math, json, random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# ---- Paths (edit if needed) ----
# Walk up from the notebook directory to find the project root (contains src/)
project_root = Path.cwd()
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / "src").is_dir():
        project_root = _p
        break

DATA_PATH = project_root / "data" / "raw" / "IEEE_PHM" / "dataset"
CACHE_DIR = project_root / "data" / "processed" / "ieee_phm_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("project_root:", project_root)
print("DATA_PATH:", DATA_PATH, "| exists:", DATA_PATH.exists())
print("CACHE_DIR:", CACHE_DIR)

project_root: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel
DATA_PATH: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/data/raw/IEEE_PHM/dataset | exists: True
CACHE_DIR: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/data/processed/ieee_phm_cache


In [8]:
# ====== 1) Config ======
_ACC_RE = re.compile(r"acc_(\d+)\.csv$", re.IGNORECASE)
_TEMP_RE = re.compile(r"temp_(\d+)\.csv$", re.IGNORECASE)

@dataclass
class PHMConfig:
    root_dir: str = str(DATA_PATH)
    seq_len: int = 32
    stride_train: int = 4
    stride_eval: int = 1
    batch_size: int = 64

    vibration_fs: float = 25600.0
    record_interval_sec: float = 10.0

    add_fft: bool = True
    fft_bands_hz: Tuple[Tuple[float, float], ...] = (
        (0, 500), (500, 1000), (1000, 2500),
        (2500, 5000), (5000, 10000), (10000, 12800),
    )

    trend_windows: Tuple[int, ...] = (10, 30)
    baseline_n: int = 10

    rul_cap_sec: float = 1500.0
    target_mode: str = "log1p_standard"

    use_sample_weights: bool = True
    weight_tau: float = 400.0
    weight_alpha: float = 2.0

    use_health_index: bool = True
    hi_topk_features: int = 12

    train_bearings: List[str] = field(default_factory=lambda: [
        "Bearing1_1", "Bearing1_2", "Bearing2_1", "Bearing2_2", "Bearing3_1", "Bearing3_2"
    ])
    val_bearings: List[str] = field(default_factory=lambda: ["Bearing1_3", "Bearing2_3"])
    test_bearings: List[str] = field(default_factory=lambda: [
        "Bearing1_4", "Bearing1_5", "Bearing1_6", "Bearing1_7",
        "Bearing2_4", "Bearing2_5", "Bearing2_6", "Bearing2_7",
        "Bearing3_3"
    ])

    seed: int = 42

cfg = PHMConfig()
seed_everything(cfg.seed)
print(cfg)

PHMConfig(root_dir='/home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/data/raw/IEEE_PHM/dataset', seq_len=32, stride_train=4, stride_eval=1, batch_size=64, vibration_fs=25600.0, record_interval_sec=10.0, add_fft=True, fft_bands_hz=((0, 500), (500, 1000), (1000, 2500), (2500, 5000), (5000, 10000), (10000, 12800)), trend_windows=(10, 30), baseline_n=10, rul_cap_sec=1500.0, target_mode='log1p_standard', use_sample_weights=True, weight_tau=400.0, weight_alpha=2.0, use_health_index=True, hi_topk_features=12, train_bearings=['Bearing1_1', 'Bearing1_2', 'Bearing2_1', 'Bearing2_2', 'Bearing3_1', 'Bearing3_2'], val_bearings=['Bearing1_3', 'Bearing2_3'], test_bearings=['Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3'], seed=42)


In [9]:
# ====== 2) Utilities: reading + robust stats ======
def safe_read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, header=None, engine="python", sep=None)

def basic_stats(x: np.ndarray, prefix: str) -> Dict[str, float]:
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        keys = ["mean","std","rms","p2p","abs_mean","skew","kurtosis","crest"]
        return {f"{prefix}_{k}": np.nan for k in keys}

    mean = float(np.mean(x))
    std  = float(np.std(x))
    rms  = float(np.sqrt(np.mean(x**2)))
    p2p  = float(np.max(x) - np.min(x))
    abs_mean = float(np.mean(np.abs(x)))

    if std < 1e-12:
        skew, kurt = 0.0, 0.0
    else:
        z = (x - mean) / std
        skew = float(np.mean(z**3))
        kurt = float(np.mean(z**4) - 3.0)

    crest = float(np.max(np.abs(x)) / (rms + 1e-12))
    return {
        f"{prefix}_mean": mean, f"{prefix}_std": std,
        f"{prefix}_rms": rms, f"{prefix}_p2p": p2p,
        f"{prefix}_abs_mean": abs_mean, f"{prefix}_skew": skew,
        f"{prefix}_kurtosis": kurt, f"{prefix}_crest": crest,
    }

def fft_band_features(x: np.ndarray, fs: float, bands, prefix: str) -> Dict[str, float]:
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        out = {f"{prefix}_total_energy": np.nan, f"{prefix}_dom_freq_hz": np.nan}
        for lo, hi in bands:
            out[f"{prefix}_band_{int(lo)}_{int(hi)}_energy"] = np.nan
        return out

    x = x - np.mean(x)
    spec = np.fft.rfft(x)
    psd = (np.abs(spec) ** 2) / max(len(x), 1)
    freqs = np.fft.rfftfreq(len(x), d=1.0 / fs)

    total = float(np.sum(psd) + 1e-12)
    out = {f"{prefix}_total_energy": total}

    for lo, hi in bands:
        mask = (freqs >= lo) & (freqs < hi)
        out[f"{prefix}_band_{int(lo)}_{int(hi)}_energy"] = float(np.sum(psd[mask]) / total)

    if len(psd) > 1:
        idx = int(np.argmax(psd[1:]) + 1)
        out[f"{prefix}_dom_freq_hz"] = float(freqs[idx])
    else:
        out[f"{prefix}_dom_freq_hz"] = 0.0

    return out

In [10]:
# ====== 3) Feature extraction per record ======
def extract_vibration_features(acc_file: Path, cfg: PHMConfig) -> Dict[str, Any]:
    df = safe_read_csv(acc_file)
    if df.shape[1] < 6:
        raise ValueError(f"Unexpected acc format: {acc_file} with shape {df.shape}")

    h = pd.to_numeric(df.iloc[:, 4], errors="coerce").to_numpy(float)
    v = pd.to_numeric(df.iloc[:, 5], errors="coerce").to_numpy(float)

    feat = {}
    feat.update(basic_stats(h, "h"))
    feat.update(basic_stats(v, "v"))

    if np.nanstd(h) > 0 and np.nanstd(v) > 0:
        feat["hv_corr"] = float(np.corrcoef(h, v)[0, 1])
    else:
        feat["hv_corr"] = 0.0
    feat["hv_rms_ratio"] = feat["h_rms"] / (feat["v_rms"] + 1e-12)

    if cfg.add_fft:
        feat.update(fft_band_features(h, cfg.vibration_fs, cfg.fft_bands_hz, "h"))
        feat.update(fft_band_features(v, cfg.vibration_fs, cfg.fft_bands_hz, "v"))

    m = _ACC_RE.search(acc_file.name)
    feat["record_idx"] = int(m.group(1)) if m else -1
    return feat

def extract_temperature(temp_file: Path) -> Dict[str, Any]:
    df = safe_read_csv(temp_file)
    if df.shape[1] < 5:
        raise ValueError(f"Unexpected temp format: {temp_file} with shape {df.shape}")

    temp = pd.to_numeric(df.iloc[:, 4], errors="coerce").to_numpy(float)
    m = _TEMP_RE.search(temp_file.name)
    return {
        "temp_record_idx": int(m.group(1)) if m else -1,
        "temp_mean": float(np.nanmean(temp)) if len(temp) else np.nan,
        "temp_std": float(np.nanstd(temp)) if len(temp) else np.nan,
        "temp_last": float(temp[-1]) if len(temp) else np.nan,
    }

In [11]:
# ====== 4) Build per-bearing table with caching ======
def condition_id_from_bearing(bearing_name: str) -> int:
    m = re.match(r"Bearing(\d+)_\d+", bearing_name)
    return int(m.group(1)) if m else -1

def add_trend_features(df: pd.DataFrame, cfg: PHMConfig) -> pd.DataFrame:
    cols_for_trend = [c for c in df.columns if any(c.endswith(s) for s in ["_rms","_kurtosis","_crest","_p2p"])]
    cols_for_trend += ["temp_mean"] if "temp_mean" in df.columns else []

    for feat in cols_for_trend:
        if feat not in df.columns:
            continue
        col = df[feat]
        for w in cfg.trend_windows:
            df[f"{feat}_rmean{w}"] = col.rolling(w, min_periods=1).mean()
            df[f"{feat}_rstd{w}"] = col.rolling(w, min_periods=1).std().fillna(0.0)
        df[f"{feat}_delta"] = col.diff().fillna(0.0)
        df[f"{feat}_ema30"] = col.ewm(span=30, min_periods=1).mean()

        baseline = col.iloc[:min(cfg.baseline_n, len(df))].mean()
        df[f"{feat}_rel_base"] = (col - baseline) / (abs(baseline) + 1e-12)

    for cond in [1,2,3]:
        df[f"cond_{cond}"] = (df["condition_id"] == cond).astype(np.float32)

    df["has_temp"] = (~df["temp_mean"].isna()).astype(np.float32) if "temp_mean" in df.columns else 0.0
    return df

def build_bearing_df(bearing_dir: Path, source: str, cfg: PHMConfig) -> pd.DataFrame:
    bearing = bearing_dir.name
    acc_files = sorted([p for p in bearing_dir.glob("acc_*.csv")],
                       key=lambda p: int(_ACC_RE.search(p.name).group(1)))
    temp_files = sorted([p for p in bearing_dir.glob("temp_*.csv")],
                        key=lambda p: int(_TEMP_RE.search(p.name).group(1)))

    if not acc_files:
        return pd.DataFrame()

    rows = []
    for f in acc_files:
        row = extract_vibration_features(f, cfg)
        row["bearing_name"] = bearing
        row["source"] = source
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("record_idx").reset_index(drop=True)
    df["step_idx"] = np.arange(len(df), dtype=int)
    df["t_rel_sec"] = df["step_idx"] * cfg.record_interval_sec
    df["condition_id"] = condition_id_from_bearing(bearing)

    if temp_files:
        temp_rows = [extract_temperature(f) for f in temp_files]
        tdf = pd.DataFrame(temp_rows).sort_values("temp_record_idx").reset_index(drop=True)
        tdf["temp_t_rel_sec"] = np.arange(len(tdf)) * 60.0
        df = pd.merge_asof(
            df.sort_values("t_rel_sec"),
            tdf[["temp_t_rel_sec","temp_mean","temp_std","temp_last"]].sort_values("temp_t_rel_sec"),
            left_on="t_rel_sec", right_on="temp_t_rel_sec",
            direction="backward", allow_exact_matches=True
        )
        for c in ["temp_mean","temp_std","temp_last"]:
            df[c] = df[c].ffill().bfill()
    else:
        df["temp_mean"] = np.nan
        df["temp_std"]  = np.nan
        df["temp_last"] = np.nan

    df = add_trend_features(df, cfg)

    if source in {"Learning_set", "Full_Test_Set"}:
        max_t = float(df["t_rel_sec"].max())
        df["rul_sec"] = max_t - df["t_rel_sec"]
    else:
        df["rul_sec"] = np.nan

    return df

def build_full_table(cfg: PHMConfig, force_rebuild: bool = False) -> pd.DataFrame:
    cache_file = CACHE_DIR / f"phm2012_features_fft{int(cfg.add_fft)}.parquet"
    if cache_file.exists() and not force_rebuild:
        print("Loading cached features:", cache_file)
        return pd.read_parquet(cache_file)

    root = Path(cfg.root_dir)
    if not root.exists():
        raise FileNotFoundError(
            f"Dataset root not found: {root}\n"
            f"Please check DATA_PATH. Current working directory is {Path.cwd()}"
        )

    frames = []
    for source in ["Learning_set", "Full_Test_Set", "Test_set"]:
        src_dir = root / source
        if not src_dir.exists():
            continue
        for bdir in sorted([x for x in src_dir.iterdir() if x.is_dir()]):
            print(f"Processing {source}/{bdir.name} ...")
            bdf = build_bearing_df(bdir, source, cfg)
            if not bdf.empty:
                frames.append(bdf)
                
    print(type(frames), len(frames))

    if len(frames) == 0:
        raise ValueError(
            f"No bearing data found under {root}. "
            f"Expected sub-folders: Learning_set/, Full_Test_Set/, Test_set/ "
            f"each containing BearingX_Y directories with acc_*.csv files."
        )

    full_df = pd.concat(frames, ignore_index=True)
    full_df = full_df.sort_values(["source","bearing_name","step_idx"]).reset_index(drop=True)
    full_df.to_parquet(cache_file, index=False)
    print("Saved cache:", cache_file)
    return full_df

full_df = build_full_table(cfg, force_rebuild=False)
print("Rows:", len(full_df), "Bearings:", full_df["bearing_name"].nunique(), "Cols:", full_df.shape[1])

Processing Learning_set/Bearing1_1 ...
Processing Learning_set/Bearing1_2 ...
Processing Learning_set/Bearing2_1 ...
Processing Learning_set/Bearing2_2 ...
Processing Learning_set/Bearing3_1 ...
Processing Learning_set/Bearing3_2 ...
Processing Full_Test_Set/Bearing1_3 ...
Processing Full_Test_Set/Bearing1_4 ...
Processing Full_Test_Set/Bearing1_5 ...
Processing Full_Test_Set/Bearing1_6 ...
Processing Full_Test_Set/Bearing1_7 ...
Processing Full_Test_Set/Bearing2_3 ...
Processing Full_Test_Set/Bearing2_4 ...
Processing Full_Test_Set/Bearing2_5 ...
Processing Full_Test_Set/Bearing2_6 ...
Processing Full_Test_Set/Bearing2_7 ...
Processing Full_Test_Set/Bearing3_3 ...
Processing Test_set/Bearing1_3 ...
Processing Test_set/Bearing1_4 ...
Processing Test_set/Bearing1_5 ...
Processing Test_set/Bearing1_6 ...
Processing Test_set/Bearing1_7 ...
Processing Test_set/Bearing2_3 ...
Processing Test_set/Bearing2_4 ...
Processing Test_set/Bearing2_5 ...
Processing Test_set/Bearing2_6 ...
Processing 

In [12]:
# ====== 5) Feature set definition (NO leakage) ======
EXCLUDE_COLS = {
    "bearing_name", "source", "record_idx", "step_idx",
    "t_rel_sec", "condition_id",
    "rul_sec",
    "temp_t_rel_sec", "temp_record_idx",
}

num_cols = full_df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in num_cols if c not in EXCLUDE_COLS]

nan_frac = full_df[feature_cols].isna().mean()
drop_cols = nan_frac[nan_frac > 0.5].index.tolist()
feature_cols = [c for c in feature_cols if c not in drop_cols]

print("Num features:", len(feature_cols))
print("Dropped (mostly NaN):", drop_cols)

Num features: 104
Dropped (mostly NaN): []


In [13]:
# ====== 6) Labelled split (per-bearing) ======
labelled_df = full_df[full_df["rul_sec"].notna()].copy()

train_df = labelled_df[labelled_df["bearing_name"].isin(cfg.train_bearings)].copy()
val_df   = labelled_df[labelled_df["bearing_name"].isin(cfg.val_bearings)].copy()
test_df  = labelled_df[labelled_df["bearing_name"].isin(cfg.test_bearings)].copy()

print("Train bearings:", sorted(train_df["bearing_name"].unique()))
print("Val bearings  :", sorted(val_df["bearing_name"].unique()))
print("Test bearings :", sorted(test_df["bearing_name"].unique()))
print("Train rows:", len(train_df), "Val rows:", len(val_df), "Test rows:", len(test_df))

Train bearings: ['Bearing1_1', 'Bearing1_2', 'Bearing2_1', 'Bearing2_2', 'Bearing3_1', 'Bearing3_2']
Val bearings  : ['Bearing1_3', 'Bearing2_3']
Test bearings : ['Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3']
Train rows: 7534 Val rows: 4330 Test rows: 13025


In [14]:
# ====== 7) Optional: Feature scoring + Health Index (HI) ======
def monotonicity_score(x: np.ndarray) -> float:
    dx = np.diff(x)
    if len(dx) == 0:
        return 0.0
    pos = np.sum(dx > 0)
    neg = np.sum(dx < 0)
    return abs(pos - neg) / max(len(dx), 1)

def correlation_to_time(x: np.ndarray) -> float:
    if len(x) < 3 or np.std(x) < 1e-12:
        return 0.0
    t = np.arange(len(x), dtype=float)
    return abs(np.corrcoef(t, x)[0,1])

def robustness_score(x: np.ndarray, w: int = 5) -> float:
    if len(x) < 3:
        return 0.0
    s = pd.Series(x).rolling(w, min_periods=1).mean().to_numpy()
    r = x - s
    denom = np.std(x) + 1e-12
    return float(1.0 - min(np.std(r)/denom, 1.0))

def feature_sensitivity_table(df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
    rows = []
    for feat in feature_cols:
        scores = []
        for _, bdf in df.groupby("bearing_name"):
            x = bdf.sort_values("step_idx")[feat].to_numpy(dtype=float)
            med = np.nanmedian(x) if np.isfinite(np.nanmedian(x)) else 0.0
            x = np.nan_to_num(x, nan=med)
            scores.append((monotonicity_score(x), correlation_to_time(x), robustness_score(x)))
        m = float(np.mean([s[0] for s in scores]))
        c = float(np.mean([s[1] for s in scores]))
        r = float(np.mean([s[2] for s in scores]))
        rows.append({"feature": feat, "monotonicity": m, "time_corr": c, "robustness": r, "composite": (m+c+r)/3.0})
    return pd.DataFrame(rows).sort_values("composite", ascending=False).reset_index(drop=True)

sens_tbl = feature_sensitivity_table(train_df, feature_cols)
display(sens_tbl.head(20))

if cfg.use_health_index:
    topk = sens_tbl.head(cfg.hi_topk_features)["feature"].tolist()

    scaler_hi = StandardScaler()
    Xtr = scaler_hi.fit_transform(train_df[topk].fillna(0.0).to_numpy())
    pca = PCA(n_components=1, random_state=cfg.seed).fit(Xtr)

    def add_hi(df: pd.DataFrame) -> pd.DataFrame:
        X = scaler_hi.transform(df[topk].fillna(0.0).to_numpy())
        hi = pca.transform(X).reshape(-1)
        out = df.copy()
        out["HI_raw"] = hi
        out["HI"] = 0.0
        for b, bdf in out.groupby("bearing_name"):
            x = bdf["HI_raw"].to_numpy()
            out.loc[bdf.index, "HI"] = (x - x.min()) / (x.max() - x.min() + 1e-12)
        return out

    train_df = add_hi(train_df)
    val_df   = add_hi(val_df)
    test_df  = add_hi(test_df)

    feature_cols_model = [c for c in ["HI","has_temp","cond_1","cond_2","cond_3"] if c in train_df.columns]
else:
    feature_cols_model = feature_cols

print("Final model features:", feature_cols_model)

/tmp/ipykernel_555703/3096915473.py:30: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(x) if np.isfinite(np.nanmedian(x)) else 0.0
/tmp/ipykernel_555703/3096915473.py:30: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(x) if np.isfinite(np.nanmedian(x)) else 0.0
/tmp/ipykernel_555703/3096915473.py:30: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(x) if np.isfinite(np.nanmedian(x)) else 0.0
/tmp/ipykernel_555703/3096915473.py:30: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(x) if np.isfinite(np.nanmedian(x)) else 0.0
/tmp/ipykernel_555703/3096915473.py:30: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(x) if np.isfinite(np.nanmedian(x)) else 0.0
/tmp/ipykernel_555703/3096915473.py:30: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(x) if np.isfinite(np.nanmedian(x)) else 0.0
/tmp/ipykernel_555703/3096915473.py:30: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(x) if np.isfini

,feature,monotonicity,time_corr,robustness,composite
0,temp_mean_rmean30,0.316243,0.569536,0.990049,0.625276
1,temp_mean_ema30,0.305616,0.571606,0.989649,0.622290
2,temp_mean_rmean10,0.260300,0.566444,0.986125,0.604289
3,h_crest_ema30,0.175959,0.581636,0.840849,0.532815
4,temp_mean_rel_base,0.041710,0.566518,0.977900,0.528709
5,temp_mean,0.041710,0.566518,0.977900,0.528709
6,temp_last,0.038220,0.567645,0.978627,0.528164
7,h_kurtosis_ema30,0.265170,0.491289,0.810515,0.522325
8,h_band_2500_5000_energy,0.016523,0.715910,0.820218,0.517550
9,v_p2p_ema30,0.203584,0.456468,0.876108,0.512053


Final model features: ['HI', 'has_temp', 'cond_1', 'cond_2', 'cond_3']


In [15]:
# ====== 8) Scalers (train-only) ======
class FeatureScaler:
    def __init__(self, feature_cols: List[str]):
        self.feature_cols = feature_cols
        self.mean_ = None
        self.std_ = None
    def fit(self, df: pd.DataFrame):
        x = df[self.feature_cols].to_numpy(dtype=np.float64)
        x = np.nan_to_num(x, nan=0.0)
        self.mean_ = np.mean(x, axis=0)
        self.std_ = np.std(x, axis=0)
        self.std_[self.std_ < 1e-12] = 1.0
        return self
    def transform(self, x: np.ndarray) -> np.ndarray:
        x = np.nan_to_num(np.asarray(x, dtype=np.float64), nan=0.0)
        return (x - self.mean_) / self.std_

class TargetScaler:
    def __init__(self, mode="log1p_standard", rul_cap: Optional[float]=None):
        self.mode = mode
        self.rul_cap = rul_cap
        self.mean_ = 0.0
        self.std_ = 1.0
    def fit(self, y: np.ndarray):
        y = np.asarray(y, dtype=np.float64)
        if self.rul_cap is not None:
            y = np.minimum(y, self.rul_cap)
        y = np.clip(y, 0, None)
        if self.mode == "log1p_standard":
            y = np.log1p(y)
        self.mean_ = float(np.mean(y))
        self.std_  = float(np.std(y)) if float(np.std(y)) > 1e-12 else 1.0
        return self
    def transform(self, y: np.ndarray) -> np.ndarray:
        y = np.asarray(y, dtype=np.float64)
        if self.rul_cap is not None:
            y = np.minimum(y, self.rul_cap)
        y = np.clip(y, 0, None)
        if self.mode == "log1p_standard":
            y = np.log1p(y)
        return (y - self.mean_) / self.std_
    def inverse_transform(self, z: np.ndarray) -> np.ndarray:
        z = np.asarray(z, dtype=np.float64)
        y = z * self.std_ + self.mean_
        if self.mode == "log1p_standard":
            y = np.expm1(y)
        return np.maximum(y, 0.0)

feat_scaler = FeatureScaler(feature_cols_model).fit(train_df)
tgt_scaler  = TargetScaler(cfg.target_mode, cfg.rul_cap_sec).fit(train_df["rul_sec"].to_numpy())
print("Target scaler mean/std:", tgt_scaler.mean_, tgt_scaler.std_)

Target scaler mean/std: 7.191690596197273 0.49538676424181366


In [16]:
# ====== 9) Dataset: window embedding + optional weighting ======
class BearingWindowDataset(Dataset):
    def __init__(
        self, df: pd.DataFrame, bearing_list: List[str], feature_cols: List[str],
        seq_len: int, stride: int, feat_scaler: FeatureScaler, tgt_scaler: TargetScaler,
        use_weights: bool, tau: float, alpha: float, pad_mode: str = "edge"
    ):
        self.samples = []
        for b in bearing_list:
            bdf = df[df["bearing_name"] == b].sort_values("step_idx").reset_index(drop=True)
            if bdf.empty:
                continue
            X = feat_scaler.transform(bdf[feature_cols].to_numpy(dtype=np.float64)).astype(np.float32)
            y = bdf["rul_sec"].to_numpy(dtype=np.float64)
            n = len(bdf)
            for end in range(0, n, stride):
                if not np.isfinite(y[end]):
                    continue
                start = max(0, end - seq_len + 1)
                wdw = X[start:end+1]
                if wdw.shape[0] < seq_len:
                    need = seq_len - wdw.shape[0]
                    pad = np.repeat(wdw[:1], need, axis=0) if pad_mode == "edge" else np.zeros((need, wdw.shape[1]), np.float32)
                    wdw = np.concatenate([pad, wdw], axis=0)
                y_norm = float(tgt_scaler.transform(np.array([y[end]]))[0])
                w = float(1.0 + alpha * math.exp(-float(y[end]) / tau)) if use_weights else 1.0
                self.samples.append((wdw, y_norm, w, b, int(bdf.loc[end,"step_idx"]), float(y[end])))
        print(f"Built {len(self.samples)} samples from {len(bearing_list)} bearings")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        wdw, y_norm, w, b, step, y_raw = self.samples[idx]
        return (
            torch.from_numpy(wdw),
            torch.tensor([y_norm], dtype=torch.float32),
            torch.tensor([w], dtype=torch.float32),
            b,
            torch.tensor([step], dtype=torch.long),
            torch.tensor([y_raw], dtype=torch.float32),
        )

train_ds = BearingWindowDataset(train_df, cfg.train_bearings, feature_cols_model, cfg.seq_len, cfg.stride_train,
                               feat_scaler, tgt_scaler, cfg.use_sample_weights, cfg.weight_tau, cfg.weight_alpha)
val_ds   = BearingWindowDataset(val_df, cfg.val_bearings, feature_cols_model, cfg.seq_len, cfg.stride_eval,
                               feat_scaler, tgt_scaler, False, cfg.weight_tau, cfg.weight_alpha)
test_ds  = BearingWindowDataset(test_df, cfg.test_bearings, feature_cols_model, cfg.seq_len, cfg.stride_eval,
                               feat_scaler, tgt_scaler, False, cfg.weight_tau, cfg.weight_alpha)

g = torch.Generator().manual_seed(cfg.seed)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, generator=g, num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

xb, yb, wb, _, _, _ = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape, wb.shape)

Built 1886 samples from 6 bearings
Built 4330 samples from 2 bearings
Built 13025 samples from 9 bearings
Batch: torch.Size([64, 32, 5]) torch.Size([64, 1]) torch.Size([64, 1])


In [17]:
# ====== 10) Metrics: RMSE/MAE + IEEE PHM score ======
def rmse(yhat: np.ndarray, y: np.ndarray) -> float:
    return float(np.sqrt(np.mean((yhat - y)**2)))

def mae(yhat: np.ndarray, y: np.ndarray) -> float:
    return float(np.mean(np.abs(yhat - y)))

def phm_score(yhat: np.ndarray, y: np.ndarray) -> float:
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    eps = 1e-12
    Er = (y - yhat) / (y + eps) * 100.0
    ln05 = math.log(0.5)
    A = np.empty_like(Er)
    A[Er <= 0] = np.exp(-ln05 * (Er[Er <= 0] / 5.0))
    A[Er > 0]  = np.exp(+ln05 * (Er[Er > 0] / 20.0))
    return float(np.mean(A))

In [19]:
# ====== 11) Model: TCN + Transformer + fusion ======
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, k, dilation=1, dropout=0.0):
        super().__init__()
        self.pad = (k - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=k, dilation=dilation)
        self.norm = nn.BatchNorm1d(out_ch)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = nn.functional.pad(x, (self.pad, 0))
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        x = self.drop(x)
        return x

class TCNBlock(nn.Module):
    def __init__(self, ch, k, dilation, dropout):
        super().__init__()
        self.c1 = CausalConv1d(ch, ch, k, dilation=dilation, dropout=dropout)
        self.c2 = CausalConv1d(ch, ch, k, dilation=dilation, dropout=dropout)
    def forward(self, x):
        return x + self.c2(self.c1(x))

class TCNEncoder(nn.Module):
    def __init__(self, in_features, d_model=64, levels=4, k=3, dropout=0.2):
        super().__init__()
        self.in_proj = nn.Conv1d(in_features, d_model, kernel_size=1)
        self.blocks = nn.Sequential(*[TCNBlock(d_model, k, 2**i, dropout) for i in range(levels)])
        self.out_norm = nn.BatchNorm1d(d_model)
    def forward(self, x):
        h = self.in_proj(x.transpose(1,2))
        h = self.blocks(h)
        h = self.out_norm(h)
        return h.transpose(1,2)

class TransformerEncoder(nn.Module):
    def __init__(self, in_features, d_model=64, nhead=4, layers=2, ff=256, dropout=0.2, max_len=512):
        super().__init__()
        self.in_proj = nn.Linear(in_features, d_model)
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=ff,
                                               dropout=dropout, batch_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        h = self.in_proj(x)
        h = h + self.pos[:, :h.size(1), :]
        return self.norm(self.enc(h))

class FusionCrossAttention(nn.Module):
    def __init__(self, d_model=64, nhead=4, dropout=0.2):
        super().__init__()
        self.attn_tcn_to_tr = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.attn_tr_to_tcn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.n1 = nn.LayerNorm(d_model)
        self.n2 = nn.LayerNorm(d_model)
    def forward(self, tcn_seq, tr_seq):
        tr2, _ = self.attn_tcn_to_tr(tr_seq, tcn_seq, tcn_seq, need_weights=False)
        tr_out = self.n1(tr_seq + tr2)
        tcn2, _ = self.attn_tr_to_tcn(tcn_seq, tr_seq, tr_seq, need_weights=False)
        tcn_out = self.n2(tcn_seq + tcn2)
        return tcn_out, tr_out

class TCNTransformerRUL(nn.Module):
    def __init__(self, in_features, d_model=64, nhead=4, dropout=0.2):
        super().__init__()
        self.tcn = TCNEncoder(in_features, d_model=d_model, levels=4, k=3, dropout=dropout)
        self.tr  = TransformerEncoder(in_features, d_model=d_model, nhead=nhead, layers=2, ff=256, dropout=dropout)
        self.fuse = FusionCrossAttention(d_model=d_model, nhead=nhead, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(d_model*2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
    def forward(self, x):
        ht = self.tcn(x)
        hz = self.tr(x)
        ht, hz = self.fuse(ht, hz)
        h = torch.cat([ht.mean(dim=1), hz.mean(dim=1)], dim=-1)
        return self.head(h)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")  # Force CPU for testing
model = TCNTransformerRUL(in_features=len(feature_cols_model), d_model=64, nhead=4, dropout=0.2).to(device)
print("Params:", sum(p.numel() for p in model.parameters()))
print("Device:", device)

Params: 283777
Device: cpu


In [20]:
# ====== 12) Loss + training loop ======
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class WeightedHuber(nn.Module):
    def __init__(self, delta: float = 1.0):
        super().__init__()
        self.delta = delta
    def forward(self, pred, target, weight):
        err = pred - target
        abs_err = torch.abs(err)
        quad = torch.minimum(abs_err, torch.tensor(self.delta, device=pred.device))
        lin  = abs_err - quad
        loss = 0.5 * quad**2 + self.delta * lin
        return torch.mean(loss * weight)

criterion = WeightedHuber(delta=1.0)

def run_epoch(model, loader, train: bool, opt=None):
    model.train(train)
    total = 0.0
    preds, trues = [], []
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))

    for xb, yb, wb, _, _, yraw in loader:
        xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
        if train:
            opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            pred = model(xb)
            w_use = wb if train else torch.ones_like(wb)
            loss = criterion(pred, yb, w_use)
        if train:
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        total += float(loss.detach().cpu().item()) * xb.size(0)
        preds.append(tgt_scaler.inverse_transform(pred.detach().cpu().numpy().reshape(-1)))
        trues.append(yraw.detach().cpu().numpy().reshape(-1))

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    return {
        "loss": total / max(len(loader.dataset), 1),
        "rmse": rmse(preds, trues),
        "mae":  mae(preds, trues),
        "score": phm_score(preds, trues),
    }

opt = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
sched = CosineAnnealingLR(opt, T_max=20, eta_min=1e-5)

best_rmse = float("inf")
best_state = None
patience = 8
bad = 0

for epoch in range(1, 41):
    tr = run_epoch(model, train_loader, train=True, opt=opt)
    va = run_epoch(model, val_loader, train=False, opt=None)
    sched.step()
    print(f"Epoch {epoch:02d} | "
          f"train rmse={tr['rmse']:.1f} mae={tr['mae']:.1f} score={tr['score']:.3f} | "
          f"val rmse={va['rmse']:.1f} mae={va['mae']:.1f} score={va['score']:.3f}")

    if va["rmse"] < best_rmse:
        best_rmse = va["rmse"]
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

/tmp/ipykernel_555703/1203963365.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
/tmp/ipykernel_555703/1203963365.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):


Epoch 01 | train rmse=10124.4 mae=7382.8 score=0.101 | val rmse=11417.2 mae=9551.4 score=0.080
Epoch 02 | train rmse=10155.4 mae=7399.9 score=0.098 | val rmse=11439.3 mae=9575.1 score=0.079
Epoch 03 | train rmse=10142.0 mae=7389.0 score=0.097 | val rmse=11432.3 mae=9564.4 score=0.079
Epoch 04 | train rmse=10109.3 mae=7352.4 score=0.098 | val rmse=11431.8 mae=9561.4 score=0.079
Epoch 05 | train rmse=10138.9 mae=7382.2 score=0.096 | val rmse=11454.8 mae=9586.2 score=0.079
Epoch 06 | train rmse=10125.7 mae=7360.3 score=0.100 | val rmse=11401.9 mae=9526.4 score=0.080
Epoch 07 | train rmse=10109.8 mae=7348.1 score=0.102 | val rmse=11433.0 mae=9560.0 score=0.079
Epoch 08 | train rmse=10130.7 mae=7366.9 score=0.099 | val rmse=11393.9 mae=9511.7 score=0.081
Epoch 09 | train rmse=10123.1 mae=7344.3 score=0.100 | val rmse=11447.8 mae=9572.8 score=0.079
Epoch 10 | train rmse=10104.2 mae=7328.5 score=0.102 | val rmse=11446.7 mae=9570.5 score=0.079
Epoch 11 | train rmse=10098.0 mae=7327.4 score=0.1

In [22]:
# ====== 13) Final evaluation ======
te = run_epoch(model, test_loader, train=False, opt=None)
print("TEST:", te)

def predict_per_sample(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for xb, _, _, bname, step, yraw in loader:
            xb = xb.to(device)
            pred = model(xb)
            pred_sec = tgt_scaler.inverse_transform(pred.detach().cpu().numpy().reshape(-1))
            for i in range(len(pred_sec)):
                rows.append({
                    "bearing": bname[i],
                    "step_idx": int(step[i].item()),
                    "y_true_sec": float(yraw[i].item()),
                    "y_pred_sec": float(pred_sec[i]),
                })
    return pd.DataFrame(rows).sort_values(["bearing","step_idx"])

pred_df = predict_per_sample(model, test_loader)
display(pred_df.head())

/tmp/ipykernel_555703/1203963365.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
/tmp/ipykernel_555703/1203963365.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):


TEST: {'loss': 0.20668660513094972, 'rmse': 10969.155607483963, 'mae': 8625.810916148062, 'score': 0.0972347390094518}


,bearing,step_idx,y_true_sec,y_pred_sec
0,Bearing1_4,0,14270.0,1477.656018
1,Bearing1_4,1,14260.0,1477.645266
2,Bearing1_4,2,14250.0,1477.639656
3,Bearing1_4,3,14240.0,1477.612903
4,Bearing1_4,4,14230.0,1477.585617


## Notes / knobs that matter most

- If results are still weak, the biggest levers are:
  1. `cfg.use_health_index=True` (HI improves cross-bearing generalisation)
  2. sample weighting (`use_sample_weights`, `weight_tau`, `weight_alpha`) so early-life windows do not dominate training
  3. increase `seq_len` to 48–64 **only** if you keep `stride_train >= 4`.

- For strict “truncated bearing” realism, normalise HI using a **running** min/max rather than full-life min/max.